In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2" # Cambia el 1 por el id de la GPU que quieras usar

In [2]:
import torch
torch.cuda.is_available(), torch.cuda.device_count(), torch.cuda.get_device_name(0)

(True, 1, 'NVIDIA GeForce RTX 3090')

In [3]:
import pandas as pd
from datasets import load_dataset


DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.bfloat16 if DEVICE == 'cuda' else torch.float32
TOTAL_SAMPLES = 10000 # Ajustable según capacidad
SAMPLES_PER_CLASS = TOTAL_SAMPLES // 2
MAX_LENGTH = 512
SEED = 41
RANDOM_STATE = 41
BATCH_SIZE= 4

# Modelos
MODEL_NAMES = {
    'LLaDA': 'GSAI-ML/LLaDA-8B-Base',
    'GPT2': 'gpt2-large',
    'LLaMA': 'NousResearch/Llama-2-7b-hf',
    'BERT': 'bert-base-uncased',
    'RoBERTa': 'roberta-base',
    'GPT3': 'EleutherAI/gpt-neo-2.7B'
}



print("Cargando dataset con estructura source/text...")

# Usamos este dataset que encaja con el formato que mencionas
dataset = load_dataset("artem9k/ai-text-detection-pile", split="train")
df_raw = pd.DataFrame(dataset)

# 1. Adaptación a tu formato: 'source' y 'text'
df_full = df_raw.copy()

# Tu lógica: 1 si es 'human', 0 si es 'ai' (o cualquier otro valor de IA)
df_full['label'] = df_full['source'].apply(
    lambda x: 1 if x == 'human' else 0
)

# Creamos la columna de clase para visualización
df_full['Clase_Real'] = df_full['label'].apply(
    lambda x: 'Humano' if x == 1 else 'IA'
)

# 2. Limpieza de texto (Columna 'text')
df_full['text_cleaned'] = (
    df_full['text']
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

# Eliminamos posibles nulos en el texto
df_full = df_full.dropna(subset=['text_cleaned']).reset_index(drop=True)

# 3. Separación por clases
df_humano = df_full[df_full['Clase_Real'] == 'Humano']
df_ia = df_full[df_full['Clase_Real'] == 'IA']

print(f"Muestras encontradas -> Humanos: {len(df_humano)} | IA: {len(df_ia)}")

# 4. Muestreo y Balanceo
assert len(df_humano) >= SAMPLES_PER_CLASS, "No hay suficientes textos humanos"
assert len(df_ia) >= SAMPLES_PER_CLASS, "No hay suficientes textos de IA"

df_humano_sample = df_humano.sample(n=SAMPLES_PER_CLASS, random_state=RANDOM_STATE)
df_ia_sample = df_ia.sample(n=SAMPLES_PER_CLASS, random_state=RANDOM_STATE)

# Mezclamos para crear el set de 10k
df_balanced_10k = pd.concat([df_humano_sample, df_ia_sample]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
df_balanced_10k['Texto_ID'] = df_balanced_10k.index

print("\nDistribución final:")
print(df_balanced_10k['Clase_Real'].value_counts())
print("Total:", len(df_balanced_10k))

df_sample = df_balanced_10k


print("\nPrimeras filas del DataFrame de muestra:")
print(df_sample[['text', 'Clase_Real', 'label']].head())

texts = df_sample['text_cleaned']
labels = df_sample['Clase_Real'].values

print(f"Dataset cargado: {len(df_sample)} textos")

# Contenedores de resultados
performance_data = []
all_results = []

Cargando dataset con estructura source/text...
Muestras encontradas -> Humanos: 1028146 | IA: 364376

Distribución final:
Clase_Real
IA        5000
Humano    5000
Name: count, dtype: int64
Total: 10000

Primeras filas del DataFrame de muestra:
                                                text Clase_Real  label
0  "I feel like I've been walking around in a cag...         IA      0
1  The old man had completely black, wrinkled ski...     Humano      1
2  Congressman Derek F. Ackerman sat at the ornat...     Humano      1
3  If you're a fan of Game of Thrones, the TV ser...         IA      0
4  Japanese cuisine consists of several diverse e...         IA      0
Dataset cargado: 10000 textos


In [4]:
# Función de medición de recursos
def resource_wrapper(fn, *args, device='cuda', verbose=True):
    if device == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    
    process = psutil.Process(os.getpid())
    mem_before = process.memory_info().rss / 1024**3  # en GB

    start_time = time.time()
    try:
        result = fn(*args)
    except Exception as e:
        if verbose:
            print(f"[ERROR] La función {fn.__name__} falló: {e}")
        raise e
    elapsed = time.time() - start_time

    vram_peak = torch.cuda.max_memory_allocated() / 1024**3 if device == 'cuda' else 0.0
    mem_after = process.memory_info().rss / 1024**3
    cpu_mem = mem_after - mem_before

    return result, elapsed, vram_peak, cpu_mem

In [5]:
import os
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import psutil
import pandas as pd
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, AutoModelForMaskedLM

# --- CONFIGURACIÓN ---
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# --- CARGA DE MODELOS ---
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

# LLaDA (Difusión)
tokenizer_llada = AutoTokenizer.from_pretrained(MODEL_NAMES['LLaDA'], trust_remote_code=True)
model_llada = AutoModel.from_pretrained(MODEL_NAMES['LLaDA'], quantization_config=bnb_config, trust_remote_code=True, dtype=DTYPE).eval()
if hasattr(model_llada, "tie_weights"): model_llada.tie_weights()
LLADA_DEVICE = next(model_llada.parameters()).device

# GPT-2
tokenizer_gpt = AutoTokenizer.from_pretrained(MODEL_NAMES['GPT2'])
if tokenizer_gpt.pad_token is None: tokenizer_gpt.pad_token = tokenizer_gpt.eos_token
model_gpt = AutoModelForCausalLM.from_pretrained(MODEL_NAMES['GPT2'], dtype=DTYPE).to(DEVICE).eval()

# LLaMA
tokenizer_llama = AutoTokenizer.from_pretrained(MODEL_NAMES['LLaMA'])
model_llama = AutoModelForCausalLM.from_pretrained(MODEL_NAMES['LLaMA'], quantization_config=bnb_config, dtype=DTYPE).eval()
LLAMA_DEVICE = next(model_llama.parameters()).device

# BERT & RoBERTa (MLM)
tokenizer_bert = AutoTokenizer.from_pretrained(MODEL_NAMES['BERT'])
model_bert = AutoModelForMaskedLM.from_pretrained(MODEL_NAMES['BERT'], torch_dtype=DTYPE).to(DEVICE).eval()

tokenizer_roberta = AutoTokenizer.from_pretrained(MODEL_NAMES['RoBERTa'])
model_roberta = AutoModelForMaskedLM.from_pretrained(MODEL_NAMES['RoBERTa'], torch_dtype=DTYPE).to(DEVICE).eval()

# GPT-3 Proxy
tokenizer_gpt3 = AutoTokenizer.from_pretrained(MODEL_NAMES['GPT3'])
if tokenizer_gpt3.pad_token is None: tokenizer_gpt3.pad_token = tokenizer_gpt3.eos_token
model_gpt3 = AutoModelForCausalLM.from_pretrained(MODEL_NAMES['GPT3'], quantization_config=bnb_config, dtype=DTYPE).eval()
GPT3_DEVICE = next(model_gpt3.parameters()).device

/opt/conda/lib/python3.11/site-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)
2026-01-31 18:33:23.861201: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-31 18:33:23.989998: I tensorflow/core/platform/cpu_feature_guard.cc:210]

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


### SCORES

In [6]:
def batch_llada_scores(texts, model, tokenizer, batch_size, device):
    scores = []
    for i in tqdm(range(0, len(texts), batch_size), desc="LLaDA Scores"):
        batch = texts[i:i+batch_size].tolist()
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH).to(device)
        with torch.no_grad(), torch.amp.autocast("cuda", dtype=DTYPE):
            outputs = model(**inputs)
            logits = outputs.logits[:, :-1, :]
            labels = inputs["input_ids"][:, 1:]
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), labels.reshape(-1), reduction="none").view(labels.shape)
            scores.extend((-loss.mean(dim=1)).cpu().tolist())
    return scores

def batch_autoregressive_scores(texts, model, tokenizer, batch_size, device):
    scores = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Autoregressive Scores"):
        batch = texts[i:i+batch_size].tolist()
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH).to(device)
        with torch.no_grad(), torch.amp.autocast("cuda", dtype=DTYPE):
            outputs = model(**inputs, labels=inputs["input_ids"])
            scores.extend((-outputs.loss.detach().cpu()).repeat(len(batch)).tolist())
    return scores

def batch_mlm_scores(texts, model, tokenizer, batch_size, device):
    scores = []
    for i in tqdm(range(0, len(texts), batch_size), desc="MLM Scores"):
        batch = texts[i:i+batch_size].tolist()
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH).to(device)
        with torch.no_grad(), torch.amp.autocast("cuda", dtype=DTYPE):
            logits = model(**inputs).logits
            log_probs = torch.log_softmax(logits, dim=-1)
            ll = log_probs.gather(2, inputs["input_ids"].unsqueeze(-1)).squeeze(-1)
            ll = (ll * inputs["attention_mask"]).sum(dim=1) / inputs["attention_mask"].sum(dim=1)
            scores.extend(ll.cpu().tolist())
    return scores

In [7]:
# Inicializamos el DataFrame de métricas con la info básica
df_metrics = df_sample[['text_cleaned', 'label']].copy()
df_metrics['Clase_Real'] = df_metrics['label'].apply(lambda x: 'Humano' if x == 1 else 'IA')
df_metrics['Texto_ID'] = df_metrics.index
texts = df_metrics['text_cleaned']

# Diccionario para automatizar ejecución y registro de recursos
scoring_tasks = [
    ("Score_LLaDA", batch_llada_scores, model_llada, tokenizer_llada, LLADA_DEVICE),
    ("Score_GPT", batch_autoregressive_scores, model_gpt, tokenizer_gpt, DEVICE),
    ("Score_LLAMA", batch_autoregressive_scores, model_llama, tokenizer_llama, LLAMA_DEVICE),
    ("Score_GPT3", batch_autoregressive_scores, model_gpt3, tokenizer_gpt3, GPT3_DEVICE),
    ("Score_BERT", batch_mlm_scores, model_bert, tokenizer_bert, DEVICE),
    ("Score_RoBERTa", batch_mlm_scores, model_roberta, tokenizer_roberta, DEVICE),
]

for col_name, func, model, tok, dev in scoring_tasks:
    print(f"\nCalculando {col_name}...")
    result, t, v, ram = resource_wrapper(func, texts, model, tok, BATCH_SIZE, dev)
    df_metrics[col_name] = result
    performance_data.append({
        "Modelo": col_name.split('_')[1],
        "Enfoque": "Sequence Score",
        "Tiempo (s)": t,
        "VRAM Pico (GB)": v,
        "RAM usada (GB)": ram
    })

torch.cuda.empty_cache()
print("\n--- Enfoque 1 completado ---")
df_metrics.head()


Calculando Score_LLaDA...


LLaDA Scores: 100%|██████████| 2500/2500 [30:19<00:00,  1.37it/s]



Calculando Score_GPT...


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
Autoregressive Scores: 100%|██████████| 2500/2500 [06:52<00:00,  6.05it/s]



Calculando Score_LLAMA...


...

MLM Scores: 100%|██████████| 2500/2500 [01:32<00:00, 27.11it/s]



Calculando Score_RoBERTa...


MLM Scores: 100%|██████████| 2500/2500 [01:37<00:00, 25.77it/s]



--- Enfoque 1 completado ---


,text_cleaned,label,Clase_Real,Texto_ID,Score_LLaDA,Score_GPT,Score_LLAMA,Score_GPT3,Score_BERT,Score_RoBERTa
0,"""I feel like I've been walking around in a cag...",0,IA,0,-13.342926,-5.565355,-5.074352,-4.213131,-0.101554,-0.025367
1,"The old man had completely black, wrinkled ski...",1,Humano,1,-11.463368,-5.565355,-5.074352,-4.213131,-0.253823,-0.112220
2,Congressman Derek F. Ackerman sat at the ornat...,1,Humano,2,-11.261955,-5.565355,-5.074352,-4.213131,-0.364764,-0.156924
3,"If you're a fan of Game of Thrones, the TV ser...",0,IA,3,-12.196795,-5.565355,-5.074352,-4.213131,-0.212125,-0.006747
4,Japanese cuisine consists of several diverse e...,0,IA,4,-11.504403,-6.234770,-6.120128,-3.239110,-1.190977,-0.000450


### PAWN 

In [8]:
import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm

def calculate_five_metrics(logits, labels, attention_mask):
    B, T, V = logits.shape
    log_probs = F.log_softmax(logits, dim=-1)
    probs = log_probs.exp()  # Calcular probs UNA VEZ
    
    # 1. Log-prob del token ocurrido
    log_p = log_probs.gather(2, labels.unsqueeze(-1)).squeeze(-1)
    Mlog_prob = log_p
    
    # 2. Entropía
    Mentropy = -(probs * log_probs).sum(dim=-1)
    
    # 3. Máxima log-prob
    Mmax_log_prob, _ = log_probs.max(dim=-1)
    
    # 4. Rank normalizado (CORREGIDO)

    rank = (log_probs > log_p.unsqueeze(-1)).sum(dim=-1).float() + 1.0
    Mrank = rank / V
    
    # 5. Top-p (CORREGIDO)
    prob_occured = probs.gather(2, labels.unsqueeze(-1)).squeeze(-1).unsqueeze(-1)
    Mtop_p = (probs * (probs >= prob_occured)).sum(dim=-1)
    
    metrics = [Mlog_prob, Mentropy, Mmax_log_prob, Mrank, Mtop_p]
    
    seq_len = attention_mask.sum(dim=1).clamp(min=1)
    results = []
    for M in metrics:
        results.append(((M * attention_mask).sum(dim=1) / seq_len).cpu().tolist())
    
    return results

def batch_autoregressive_metrics(texts, model, tokenizer, batch_size, device):
    """
    Calcula las 5 métricas para modelos autoregresivos (LLaDA, GPT, LLaMA, GPT-3)
    """
    all_metrics = [[], [], [], [], []] # 5 listas para las 5 métricas
    
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size].tolist()

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH
        )
        inputs_on_device = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad(), torch.cuda.amp.autocast(dtype=DTYPE):
            outputs = model(**inputs_on_device)
            
            logits = outputs.logits[:, :-1, :]        # (B, T-1, V)
            labels = inputs_on_device["input_ids"][:, 1:] # (B, T-1)

            attention_mask = inputs_on_device["attention_mask"][:, 1:] # (B, T-1)
            
            metrics = calculate_five_metrics(logits, labels, attention_mask)
            
            for j in range(5):
                all_metrics[j].extend(metrics[j])
                
    return all_metrics
def calculate_five_metrics_mlm(logits, labels, attention_mask):
    """
    Calcula las 5 métricas PAWN para modelos MLM.
    Para LLaDA en modo MLM: evalúa cada posición de la secuencia.
    """
    B, T, V = logits.shape
    
    log_probs = F.log_softmax(logits, dim=-1)  # (B, T, V)
    probs = torch.exp(log_probs)
    
    # 1. Log-prob del token real
    log_prob_occured = log_probs.gather(
        2, labels.unsqueeze(-1)
    ).squeeze(-1)  # (B, T)
    
    Mlog_prob = log_prob_occured
    
    # 2. Entropía
    entropy = -(probs * log_probs).sum(dim=-1)  # (B, T)
    Mentropy = entropy
    
    # 3. Max log-prob
    Mmax_log_prob, _ = log_probs.max(dim=-1)  # (B, T)
    
    # 4. Rank (CORREGIDO - posición ordinal normalizada)
    log_prob_occured_val = log_prob_occured.unsqueeze(-1)  # (B, T, 1)
    rank = (log_probs > log_prob_occured_val).sum(dim=-1).float() + 1.0
    Mrank = rank / V  # Normalizar
    
    # 5. Top-p
    prob_occured = probs.gather(2, labels.unsqueeze(-1)).squeeze(-1).unsqueeze(-1)
    Mtop_p = (probs * (probs >= prob_occured)).sum(dim=-1)
    
    masked_metrics = [Mlog_prob, Mentropy, Mmax_log_prob, Mrank, Mtop_p]
    results = []
    sequence_lengths = attention_mask.sum(dim=1).float().clamp(min=1)  # (B,)
    
    for M in masked_metrics:
        M_masked = M * attention_mask
        M_sum_per_seq = M_masked.sum(dim=1)
        M_avg_per_seq = M_sum_per_seq / sequence_lengths
        results.append(M_avg_per_seq.cpu().tolist())
    
    return results


def batch_mlm_metrics(texts, model, tokenizer, batch_size, device):
    """
    Calcula las 5 métricas PAWN para LLaDA en modo MLM.
    LLaDA puede funcionar como MLM bidireccional.
    """
    all_metrics = [[], [], [], [], []]
    vocab_size = model.config.vocab_size
    
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size].tolist()
        
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH
        ).to(device)
        
        input_ids = inputs["input_ids"]
        attention_mask = inputs["attention_mask"]
        
        # CRÍTICO: Clamp de seguridad
        input_ids = input_ids.clamp(0, vocab_size - 1)
        
        with torch.no_grad(), torch.amp.autocast("cuda", dtype=DTYPE):
            # LLaDA puede retornar logits para todas las posiciones
            outputs = model(**inputs)
            
            # Si LLaDA retorna logits de forma (B, T, V), úsalos directamente
            logits = outputs.logits  # (B, T, V)
            
            # Calcular las 5 métricas
            metrics = calculate_five_metrics_mlm(logits, input_ids, attention_mask)
            
            for j in range(5):
                all_metrics[j].extend(metrics[j])
    
    return all_metrics

def calculate_five_metrics_diffusion(logits, labels, mask_positions):
    """
    Calcula las 5 métricas PAWN solo en las posiciones ENMASCARADAS.
    Basado en la lógica de reconstrucción de Language Diffusion.
    """
    B, T, V = logits.shape
    
    # Trabajamos con log_softmax para estabilidad numérica
    log_probs = F.log_softmax(logits, dim=-1)
    probs = torch.exp(log_probs)
    
    # 1. Log-probabilidad del token real (¿Qué tan bien reconstruye el modelo el texto original?)
    log_p = log_probs.gather(2, labels.unsqueeze(-1)).squeeze(-1)
    
    # 2. Entropía (Incertidumbre del modelo en las zonas enmascaradas)
    entropy = -(probs * log_probs).sum(dim=-1)
    
    # 3. Max log-prob (Confianza máxima en la predicción)
    max_log_p, _ = log_probs.max(dim=-1)
    
    # 4. Rank normalizado (Posición ordinal del token real)
    # Calculamos cuántos tokens tienen mayor probabilidad que el token real
    rank = (log_probs > log_p.unsqueeze(-1)).sum(dim=-1).float() + 1.0
    rank_norm = rank / V
    
    # 5. Top-p (Masa de probabilidad acumulada necesaria para llegar al token real)
    # Indica qué tan "esperada" era la palabra en ese contexto
    prob_occured = probs.gather(2, labels.unsqueeze(-1)).squeeze(-1).unsqueeze(-1)
    top_p = (probs * (probs >= prob_occured)).sum(dim=-1)
    
    mask_float = mask_positions.float()
    num_masked = mask_float.sum(dim=1).clamp(min=1)
    
    results = []
    # Iteramos sobre las 5 métricas calculadas
    for M in [log_p, entropy, max_log_p, rank_norm, top_p]:
        # Filtramos para obtener el promedio SOLO de los tokens que fueron enmascarados
        # Esto es lo que mide la capacidad de "denoising" o reconstrucción.
        avg_val = (M * mask_float).sum(dim=1) / num_masked
        results.append(avg_val.cpu().tolist())
        
    return results

def batch_diffusion_metrics(texts, model, tokenizer, batch_size, device, mask_ratio=0.35, num_samples=10):
    """
    Implementa el proceso de scoring por difusión para LLaDA.
    Aumentamos el mask_ratio a 0.20 para forzar al modelo a usar más contexto.
    Reducimos num_samples a 3 para balancear velocidad y estabilidad.
    """
    all_metrics = [[] for _ in range(5)]
    
    # Identificar token de máscara correcto
    if hasattr(tokenizer, 'mask_token_id') and tokenizer.mask_token_id is not None:
        mask_id = tokenizer.mask_token_id
    else:
        # Fallback para modelos que no tienen [MASK] definido explícitamente
        mask_id = tokenizer.vocab_size - 1 

    model.eval()
    
    # Determinamos el tipo de dato para autocast (bfloat16 para GPUs modernas como RTX 4500 Ada)
    dtype = torch.bfloat16 if device == 'cuda' else torch.float32

    for i in tqdm(range(0, len(texts), batch_size), desc="LLaDA Diffusion Scoring"):
        batch_texts = texts[i:i+batch_size].tolist()
        # Acumulador para las muestras estocásticas de cada batch
        batch_accum = [[] for _ in range(5)]
        
        # Realizamos varias pasadas con diferentes máscaras para obtener un promedio robusto (Monte Carlo)
        for _ in range(num_samples):
            inputs = tokenizer(
                batch_texts, 
                return_tensors="pt", 
                padding=True, 
                truncation=True, 
                max_length=512
            ).to(device)
            
            input_ids = inputs["input_ids"]
            att_mask = inputs["attention_mask"]
            B, T = input_ids.shape
            
            # Generamos máscara aleatoria excluyendo el padding
            mask_probs = torch.full((B, T), mask_ratio, device=device) * att_mask.float()
            
            # Opcional: Evitar enmascarar tokens especiales (si el tokenizer los tiene definidos)
            if hasattr(tokenizer, 'all_special_ids'):
                for special_id in tokenizer.all_special_ids:
                    mask_probs[input_ids == special_id] = 0.0

            mask_pos = torch.bernoulli(mask_probs).bool()
            
            # Garantizar que al menos un token esté enmascarado por secuencia
            for j in range(B):
                if not mask_pos[j].any():
                    valid_indices = att_mask[j].nonzero(as_tuple=True)[0]
                    if len(valid_indices) > 0:
                        random_idx = valid_indices[torch.randint(0, len(valid_indices), (1,))]
                        mask_pos[j, random_idx] = True

            # Crear la versión "corrupta" del texto
            corrupted_ids = input_ids.clone()
            corrupted_ids[mask_pos] = mask_id
            
            with torch.no_grad(), torch.amp.autocast("cuda", dtype=dtype):
                # Predicción bidireccional: el modelo intenta adivinar los tokens en las máscaras
                outputs = model(input_ids=corrupted_ids, attention_mask=att_mask)
                
                # Extraer métricas solo de las posiciones que el modelo tuvo que reconstruir
                sample_m = calculate_five_metrics_diffusion(outputs.logits, input_ids, mask_pos)
                
                for m_idx in range(5):
                    batch_accum[m_idx].append(sample_m[m_idx])
        
        # Promediar las muestras para reducir el ruido de la selección aleatoria de máscaras
        for m_idx in range(5):
            avg_res = np.array(batch_accum[m_idx]).mean(axis=0)
            all_metrics[m_idx].extend(avg_res.tolist())
            
    return all_metrics

In [9]:
metrics_names = ['Mlog_prob', 'Mentropy', 'Mmax_log_prob', 'Mrank', 'Mtop_p']

# Tareas de extracción PAWN
pawn_tasks = [
    ("LLaDA", batch_diffusion_metrics, model_llada, tokenizer_llada, LLADA_DEVICE), # Usa tu función diffusion
    ("GPT", batch_autoregressive_metrics, model_gpt, tokenizer_gpt, DEVICE),
    ("LLaMA", batch_autoregressive_metrics, model_llama, tokenizer_llama, LLAMA_DEVICE),
    ("GPT3", batch_autoregressive_metrics, model_gpt3, tokenizer_gpt3, GPT3_DEVICE),
    ("BERT", batch_mlm_metrics, model_bert, tokenizer_bert, DEVICE),
    ("RoBERTa", batch_mlm_metrics, model_roberta, tokenizer_roberta, DEVICE),
]

for mod_name, func, model, tok, dev in pawn_tasks:
    print(f"\nCalculando métricas PAWN para {mod_name}...")
    # Registramos recursos
    pawn_results, t, v, ram = resource_wrapper(func, texts, model, tok, BATCH_SIZE, dev)
    
    # Guardamos las 5 métricas en el DataFrame
    for idx, m_name in enumerate(metrics_names):
        df_metrics[f'{m_name}_{mod_name}'] = pawn_results[idx]
        
    # Añadimos a la tabla de rendimiento
    performance_data.append({
        "Modelo": mod_name,
        "Enfoque": "PAWN Extraction",
        "Tiempo (s)": t,
        "VRAM Pico (GB)": v,
        "RAM usada (GB)": ram
    })

torch.cuda.empty_cache()
print("\n--- Enfoque 2: PAWN Metrics completado ---")
df_metrics.filter(like='Mlog_prob').head()


Calculando métricas PAWN para LLaDA...


LLaDA Diffusion Scoring: 100%|██████████| 2500/2500 [8:18:33<00:00, 11.97s/it]  



Calculando métricas PAWN para GPT...


/tmp/ipykernel_3765/712804160.py:57: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(dtype=DTYPE):
IOPub message rate exceeded.05:33<01:17,  5.63it/s]
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



### EMBEDINGS

In [10]:
def extract_cls_embeddings_autoregressive(texts, model, tokenizer, device, batch_size=8):
    """
    Extrae embeddings de la ÚLTIMA posición (equivalente a CLS en modelos autoregresivos).
    Estos modelos generan representaciones causales, por lo que el último token
    tiene información de toda la secuencia.
    """
    all_embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Extrayendo embeddings autoregresivos"):
        batch = texts[i:i+batch_size].tolist()
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad(), torch.amp.autocast('cuda', dtype=DTYPE):
            outputs = model(**inputs, output_hidden_states=True)
            hidden_states = outputs.hidden_states[-1]  # última capa
            
            # Extraer embedding del último token no-padding de cada secuencia
            attention_mask = inputs['attention_mask']
            seq_lengths = attention_mask.sum(dim=1) - 1  # índice del último token
            
            batch_embeddings = []
            for j, seq_len in enumerate(seq_lengths):
                # Último token con información de toda la secuencia
                embedding = hidden_states[j, seq_len, :].float().cpu().numpy()
                batch_embeddings.append(embedding)
            
            all_embeddings.extend(batch_embeddings)
    
    return np.array(all_embeddings)


def extract_cls_embeddings_mlm(texts, model, tokenizer, device, batch_size=8):
    """
    Extrae embeddings del token [CLS] o equivalente para modelos MLM.
    BERT y RoBERTa tienen un token especial al inicio que agrega contexto bidireccional.
    """
    all_embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Extrayendo embeddings MLM"):
        batch = texts[i:i+batch_size].tolist()
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad(), torch.amp.autocast('cuda', dtype=DTYPE):
            outputs = model(**inputs, output_hidden_states=True)
            hidden_states = outputs.hidden_states[-1]  # última capa
            
            # Token [CLS] está en la posición 0
            cls_embeddings = hidden_states[:, 0, :].float().cpu().numpy()
            all_embeddings.extend(cls_embeddings)
    
    return np.array(all_embeddings)


def extract_cls_embeddings_diffusion(texts, model, tokenizer, device, batch_size=8):
    """
    Extrae embeddings de LLaDA (modelo de difusión).
    LLaDA procesa bidireccionalmente, similar a BERT, por lo que usamos
    el promedio de todos los tokens como representación global.
    
    Alternativa: pooling del primer token o mean pooling de toda la secuencia.
    """
    all_embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Extrayendo embeddings difusión"):
        batch = texts[i:i+batch_size].tolist()
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad(), torch.amp.autocast('cuda', dtype=DTYPE):
            outputs = model(**inputs, output_hidden_states=True)
            hidden_states = outputs.hidden_states[-1]  # última capa
            
            # Mean pooling sobre tokens válidos (excluyendo padding)
            attention_mask = inputs['attention_mask'].unsqueeze(-1)
            masked_hidden = hidden_states * attention_mask
            sum_hidden = masked_hidden.sum(dim=1)
            count = attention_mask.sum(dim=1).clamp(min=1)
            mean_embedding = (sum_hidden / count).float().cpu().numpy()
            
            all_embeddings.extend(mean_embedding)
    
    return np.array(all_embeddings)


In [11]:
embedding_tasks = [
    ("LLaDA", extract_cls_embeddings_diffusion, model_llada, tokenizer_llada, LLADA_DEVICE),
    ("GPT2", extract_cls_embeddings_autoregressive, model_gpt, tokenizer_gpt, DEVICE),
    ("LLAMA", extract_cls_embeddings_autoregressive, model_llama, tokenizer_llama, LLAMA_DEVICE),
    ("BERT", extract_cls_embeddings_mlm, model_bert, tokenizer_bert, DEVICE),
    ("RoBERTa", extract_cls_embeddings_mlm, model_roberta, tokenizer_roberta, DEVICE)
]

for mod_name, func, model, tok, dev in embedding_tasks:
    print(f"\n[INFO] Extrayendo embeddings para {mod_name}...")
    
    # 1. Ejecutamos la extracción y medimos recursos (Tiempo, VRAM, RAM)
    # Usamos BATCH_SIZE=4 como en tus otros scripts para evitar errores de memoria
    embs_matrix, t, v, ram = resource_wrapper(func, texts, model, tok, dev, BATCH_SIZE)
    
    # 2. Guardamos los embeddings en el DataFrame
    # Convertimos la matriz numpy a una lista para que cada celda sea un vector individual
    df_metrics[f'Embedding_{mod_name}'] = list(embs_matrix)
    
    # 3. Registramos el rendimiento en la tabla global de recursos
    performance_data.append({
        "Modelo": mod_name,
        "Enfoque": "CLS Embedding Extraction",
        "Tiempo (s)": t,
        "VRAM Pico (GB)": v,
        "RAM usada (GB)": ram
    })

# Limpieza de caché de GPU
torch.cuda.empty_cache()

print("\n" + "="*50)
print("ESTADO DEL DATAFRAME INTEGRADO")
print("="*50)
# Mostramos las nuevas columnas creadas junto a las anteriores
cols_interes = ['Texto_ID'] + [c for c in df_metrics.columns if 'Embedding' in c or 'Score' in c or 'Mlog_prob' in c]
print(df_metrics[cols_interes].head())

# Guardado intermedio del progreso
df_metrics.to_csv('df_all_features_combined_DeepfakeTextDetect.csv', index=False)
print("\n✓ Todas las métricas (Scores, PAWN y Embeddings) guardadas en 'df_all_features_combined.csv'")


[INFO] Extrayendo embeddings para LLaDA...


Extrayendo embeddings difusión: 100%|██████████| 2500/2500 [30:05<00:00,  1.38it/s]



[INFO] Extrayendo embeddings para GPT2...


IOPub message rate exceeded.gresivos:  21%|██        | 531/2500 [01:21<04:58,  6.61it/s]
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [12]:
df = pd.DataFrame(performance_data)

df.to_csv('df_performance_models_proachs_DeepfakeTextDetect.csv', index=False)
df.head()

,Modelo,Enfoque,Tiempo (s),VRAM Pico (GB),RAM usada (GB)
0,LLaDA,Sequence Score,1819.677496,15.534767,-0.036022
1,GPT,Sequence Score,412.998665,14.993741,0.013710
2,LLAMA,Sequence Score,1679.826880,15.879548,-3.074024
3,GPT3,Sequence Score,832.167214,16.800107,0.005234
4,BERT,Sequence Score,92.235898,13.941291,0.006149


### CLASIFICACIÓN

In [13]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, recall_score, precision_score
from sklearn.calibration import CalibratedClassifierCV
import xgboost as xgb

# =============================================================================
# 0. FUNCIÓN DE EVALUACIÓN (idéntica a Embedings.ipynb)
# =============================================================================
def evaluate_classifier(y_true, y_pred_probs):
    y_pred = (y_pred_probs >= 0.5).astype(int)
    metrics = {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0),
    }
    try:
        metrics['ROC-AUC'] = roc_auc_score(y_true, y_pred_probs)
    except:
        metrics['ROC-AUC'] = np.nan
    return metrics

import ast

def parse_embedding_safe(x):
    """
    Convierte un string tipo '[0.1 0.2 ...]' o '[0.1, 0.2, ...]'
    en np.ndarray float32 de forma segura.
    """
    if isinstance(x, np.ndarray):
        return x
    if isinstance(x, str):
        # Normalizamos espacios -> comas
        x = x.replace('\n', ' ').replace('  ', ' ')
        if ',' not in x:
            x = x.replace(' ', ', ')
        return np.array(ast.literal_eval(x), dtype=np.float32)
    raise ValueError(f"Tipo inesperado en embedding: {type(x)}")

# =============================================================================
# 1. ARQUITECTURA Y MOTOR MLP (PAWN)
# =============================================================================
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class DeepMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 1024), nn.BatchNorm1d(1024), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 128), nn.ReLU(), nn.Linear(128, 2)
        )
    def forward(self, x): 
        return self.net(x)

def train_eval_mlp_full(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=SEED
    )
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    model = DeepMLP(X.shape[1]).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-5)
    criterion = nn.CrossEntropyLoss()

    loader = DataLoader(
        TensorDataset(torch.from_numpy(X_train).float(), torch.from_numpy(y_train).long()),
        batch_size=32, shuffle=True
    )

    for _ in range(100):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            criterion(model(xb), yb).backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        logits = model(torch.from_numpy(X_test).float().to(DEVICE))
        probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()

    return evaluate_classifier(y_test, probs)

# =============================================================================
# 2. CLASIFICACIÓN MACRO (SCORE / PAWN / EMBEDDINGS)
# =============================================================================
df = pd.read_csv("df_all_features_combined_DeepfakeTextDetect.csv")
y_all = df["label"].values
results_master = []

models_list = ['LLaDA', 'GPT2', 'LLAMA', 'GPT3', 'BERT', 'RoBERTa']
metrics_pawn_names = ['Mlog_prob', 'Mentropy', 'Mmax_log_prob', 'Mrank', 'Mtop_p']

for m in models_list:
    print(f"\nEvaluando: {m}")

    # -------------------------------------------------------------------------
    # A) SCORE
    # -------------------------------------------------------------------------
    s_col = f"Score_{m}"
    if s_col in df.columns:
        X_s = df[[s_col]].values
        clfs = {
            'LogReg': LogisticRegression(max_iter=2000),
            'RandomForest': RandomForestClassifier(n_estimators=300, random_state=SEED),
            'XGBoost': xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, eval_metric="logloss", random_state=SEED),
            'XGB_Calib': CalibratedClassifierCV(
                xgb.XGBClassifier(eval_metric="logloss"), method="isotonic", cv=3
            )
        }
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
        for name, clf in clfs.items():
            cv_res = cross_validate(
                clf, X_s, y_all, cv=cv,
                scoring=['roc_auc', 'accuracy', 'f1', 'recall', 'precision']
            )
            results_master.append({
                'Modelo': m, 'Aprox': 'Score', 'Clf': name,
                'ROC-AUC': np.mean(cv_res['test_roc_auc']),
                'Accuracy': np.mean(cv_res['test_accuracy']),
                'F1': np.mean(cv_res['test_f1']),
                'Recall': np.mean(cv_res['test_recall']),
                'Precision': np.mean(cv_res['test_precision'])
            })

    # -------------------------------------------------------------------------
    # B) PAWN
    # -------------------------------------------------------------------------
    p_cols = [f"{met}_{m}" for met in metrics_pawn_names]
    if all(c in df.columns for c in p_cols):
        X_p = df[p_cols].values

        # PAWN + LogReg
        X_tr, X_te, y_tr, y_te = train_test_split(
            X_p, y_all, test_size=0.2, stratify=y_all, random_state=SEED
        )
        sc = StandardScaler()
        clf_lr = LogisticRegression(
            max_iter=1000, class_weight='balanced', random_state=SEED
        ).fit(sc.fit_transform(X_tr), y_tr)
        p_lr = clf_lr.predict_proba(sc.transform(X_te))[:, 1]

        results_master.append({
            'Modelo': m, 'Aprox': 'PAWN', 'Clf': 'LogReg',
            **evaluate_classifier(y_te, p_lr)
        })

        # PAWN + DeepMLP
        results_master.append({
            'Modelo': m, 'Aprox': 'PAWN', 'Clf': 'DeepMLP',
            **train_eval_mlp_full(X_p, y_all)
        })

# -------------------------------------------------------------------------
# C) EMBEDDINGS (extracción + clasificación DIRECTA, sin guardar)
# -------------------------------------------------------------------------
print("\n" + "="*80)
print("EXTRACCIÓN DE EMBEDDINGS Y CLASIFICACIÓN (DIRECTO)")
print("="*80)

# Diccionario que mapea modelo → función de extracción + objetos
embedding_configs = {
    'LLaDA': {
        'extractor': lambda: extract_cls_embeddings_diffusion(
            texts, model_llada, tokenizer_llada, LLADA_DEVICE
        )
    },
    'GPT2': {
        'extractor': lambda: extract_cls_embeddings_autoregressive(
            texts, model_gpt, tokenizer_gpt, DEVICE
        )
    },
    'LLAMA': {
        'extractor': lambda: extract_cls_embeddings_autoregressive(
            texts, model_llama, tokenizer_llama, LLAMA_DEVICE
        )
    },
    'BERT': {
        'extractor': lambda: extract_cls_embeddings_mlm(
            texts, model_bert, tokenizer_bert, DEVICE
        )
    },
    'RoBERTa': {
        'extractor': lambda: extract_cls_embeddings_mlm(
            texts, model_roberta, tokenizer_roberta, DEVICE
        )
    }
}

for m, cfg in embedding_configs.items():
    print(f"\n[EMB] {m} - Extrayendo embeddings...")
    
    # 1. Extracción REAL del embedding (np.ndarray)
    X_e = cfg['extractor']()
    print(f"   Shape: {X_e.shape}")

    # 2. Split (idéntico a Embedings.ipynb)
    X_train, X_test, y_train, y_test = train_test_split(
        X_e, y_all, test_size=0.2, random_state=SEED, stratify=y_all
    )

    # 3. Escalado
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # 4. Clasificador lineal
    clf = LogisticRegression(
        max_iter=1000,
        random_state=SEED,
        class_weight='balanced'
    )
    clf.fit(X_train_scaled, y_train)

    # 5. Predicción
    y_pred_probs = clf.predict_proba(X_test_scaled)[:, 1]

    # 6. Métricas
    metrics = evaluate_classifier(y_test, y_pred_probs)

    print(f"   Resultados: {metrics}")

    # 7. Inserción DIRECTA en la macro-tabla
    results_master.append({
        'Modelo': m,
        'Aprox': 'Embedding',
        'Clf': 'LogReg',
        'ROC-AUC': metrics['ROC-AUC'],
        'Accuracy': metrics['Accuracy'],
        'F1': metrics['F1'],
        'Recall': metrics['Recall'],
        'Precision': metrics['Precision']
    })

# =============================================================================
# 3. PRESENTACIÓN DE RESULTADOS
# =============================================================================
df_final = pd.DataFrame(results_master)
print("\n" + "="*120)
print("MACRO TABLA DE RESULTADOS (COMPARATIVA FINAL)")
print("="*120)
display(df_final.set_index(['Modelo', 'Aprox', 'Clf']).round(4))



Evaluando: LLaDA

Evaluando: GPT2

Evaluando: LLAMA

Evaluando: GPT3

Evaluando: BERT

Evaluando: RoBERTa

EXTRACCIÓN DE EMBEDDINGS Y CLASIFICACIÓN (DIRECTO)

[EMB] LLaDA - Extrayendo embeddings...


Extrayendo embeddings difusión: 100%|██████████| 1250/1250 [28:40<00:00,  1.38s/it]


   Shape: (10000, 4096)
   Resultados: {'Accuracy': 0.975, 'Precision': 0.9769076305220884, 'Recall': 0.973, 'F1': 0.9749498997995992, 'ROC-AUC': 0.9962150000000001}

[EMB] GPT2 - Extrayendo embeddings...


Extrayendo embeddings autoregresivos: 100%|██████████| 1250/1250 [06:19<00:00,  3.29it/s]


   Shape: (10000, 1280)
   Resultados: {'Accuracy': 0.844, 'Precision': 0.843313373253493, 'Recall': 0.845, 'F1': 0.8441558441558441, 'ROC-AUC': 0.9173840000000001}

[EMB] LLAMA - Extrayendo embeddings...


IOPub message rate exceeded.gresivos:  39%|███▊      | 483/1250 [09:55<15:42,  1.23s/it]
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [14]:
df_final.to_csv('df_results_models_proachs_DeepfakeTextDetect.csv', index=False)

In [15]:
df_final

,Modelo,Aprox,Clf,ROC-AUC,Accuracy,F1,Recall,Precision
0,LLaDA,Score,LogReg,0.538492,0.5284,0.572552,0.6318,0.523503
1,LLaDA,Score,RandomForest,0.627960,0.5766,0.579301,0.5834,0.575497
2,LLaDA,Score,XGBoost,0.710098,0.6612,0.664551,0.6714,0.657977
3,LLaDA,Score,XGB_Calib,0.708333,0.6640,0.665518,0.6688,0.662530
4,LLaDA,PAWN,LogReg,0.853189,0.7705,0.768766,0.7630,0.774619
5,LLaDA,PAWN,DeepMLP,0.877578,0.7850,0.763996,0.6960,0.846715
6,LLAMA,Score,LogReg,0.507198,0.5102,0.507121,0.5040,0.510369
7,LLAMA,Score,RandomForest,0.498016,0.4939,0.492810,0.4918,0.493926
8,LLAMA,Score,XGBoost,0.531978,0.5313,0.536814,0.5436,0.530841
9,LLAMA,Score,XGB_Calib,0.536815,0.5299,0.526853,0.5276,0.530314
